# Model Validation

Proper cross-validation framework to choose and tune our final production model.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/model_data.csv")

X = df.drop(columns=["y"])
y = df["y"]

print("Shape:", X.shape)
print("Conversion rate:", y.mean())

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=400, max_depth=12, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1))
])

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42))
])

In [ ]:
from sklearn.model_selection import cross_validate

scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "neg_brier": "neg_brier_score"
}

logistic_cv = cross_validate(logistic_model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
rf_cv = cross_validate(rf_model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
xgb_cv = cross_validate(xgb_model, X, y, cv=cv, scoring=scoring, n_jobs=-1)

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "ROC_AUC": [
        logistic_cv["test_roc_auc"].mean(),
        rf_cv["test_roc_auc"].mean(),
        xgb_cv["test_roc_auc"].mean()
    ],
    "PR_AUC": [
        logistic_cv["test_pr_auc"].mean(),
        rf_cv["test_pr_auc"].mean(),
        xgb_cv["test_pr_auc"].mean()
    ],
    "Brier_Score": [
        -logistic_cv["test_neg_brier"].mean(),
        -rf_cv["test_neg_brier"].mean(),
        -xgb_cv["test_neg_brier"].mean()
    ]
})

results.sort_values("PR_AUC", ascending=False)

In [ ]:
validation_summary = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "PR_AUC_mean": [
        logistic_cv["test_pr_auc"].mean(),
        rf_cv["test_pr_auc"].mean(),
        xgb_cv["test_pr_auc"].mean()
    ],
    "PR_AUC_std": [
        logistic_cv["test_pr_auc"].std(),
        rf_cv["test_pr_auc"].std(),
        xgb_cv["test_pr_auc"].std()
    ]
})

validation_summary

In [ ]:
best_model_name = validation_summary.sort_values("PR_AUC_mean", ascending=False).iloc[0]["Model"]
print("Best model:", best_model_name)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "model__n_estimators": [200, 400, 600],
    "model__max_depth": [3, 5, 7, 9],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0]
}

search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=20,
    scoring="average_precision",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X, y)

print("Best score:", search.best_score_)
print("Best params:", search.best_params_)

In [ ]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../artifacts/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(search.best_estimator_, MODEL_DIR / "final_xgboost_model.joblib")

In [ ]:
EXP_DIR = Path("../artifacts/experiments")
EXP_DIR.mkdir(parents=True, exist_ok=True)

results.to_csv("../artifacts/experiments/model_results.csv", index=False)